# 6. Surface Normals and Multiple Objects

##  6.5 A List of Hittable Objects

Let’s translate this C++ `hittable_list` into Python and also understand *what it is doing mathematically and conceptually*.

This is a core idea in ray tracing: **you don’t test a ray against one object, you test it against a whole world (a list of objects), and pick the closest hit.**

---

# 🧱 1. Python Version of `HittableList`

```python
from typing import List, Optional

# We assume these exist already:
# Ray, HitRecord, Hittable

class HittableList:
    def __init__(self, object=None):
        self.objects: List["Hittable"] = []
        if object is not None:
            self.add(object)

    def clear(self):
        self.objects.clear()

    def add(self, object: "Hittable"):
        self.objects.append(object)

    def hit(self, ray, t_min: float, t_max: float, rec: "HitRecord") -> bool:
        temp_rec = HitRecord()
        hit_anything = False
        closest_so_far = t_max

        for obj in self.objects:
            if obj.hit(ray, t_min, closest_so_far, temp_rec):
                hit_anything = True
                closest_so_far = temp_rec.t
                rec.copy(temp_rec)

        return hit_anything
```

---

# 🧠 2. Big Idea (What is happening?)

We are solving this problem:

> Given a ray, which object in the scene does it hit first?

So we:

1. Test ray against every object
2. Keep only valid hits
3. Always choose the **closest one (smallest $t$)**

---

# 📌 3. Key Mathematical Idea

Each object returns solutions of:

$$
\mathbf{r}(t) = \mathbf{O} + t\mathbf{D}
$$

Where:

* $\mathbf{O}$ = ray origin
* $\mathbf{D}$ = ray direction
* $t$ = distance along ray

Each object checks:

* Does this equation intersect it?
* If yes → returns valid $t$

---

# 📊 4. Step-by-step Logic Table

| Step | Code Part                     | Math Idea                   | Purpose                         |
| ---- | ----------------------------- | --------------------------- | ------------------------------- |
| 1    | `closest_so_far = t_max`      | Upper bound of valid $t$    | Start with farthest allowed hit |
| 2    | loop over objects             | $\forall$ objects in scene  | Test every object               |
| 3    | `obj.hit(...)`                | Solve intersection equation | Find if ray hits object         |
| 4    | `temp_rec.t < closest_so_far` | $\min(t)$ selection         | Keep nearest hit                |
| 5    | `rec.copy(temp_rec)`          | store best solution         | Save closest intersection       |
| 6    | return result                 | boolean logic               | whether any hit exists          |

---

# 🧮 5. Why “closest hit” matters

If multiple objects are hit:

$$
t_1 = 2,\quad t_2 = 5,\quad t_3 = 1
$$

We choose:

$$
t_{\text{final}} = \min(t_1, t_2, t_3) = 1
$$

### Why?

Because ray travels forward — the first surface it touches is what we see.

---

# ⚙️ 6. Why we shrink `closest_so_far`

This line is the most important:

```python
closest_so_far = temp_rec.t
```

### Meaning:

We are updating the search range:

$$
t_{\text{max}} \leftarrow t_{\text{current best}}
$$

So next objects only compete with a *closer limit*.

---

# 🔁 7. Flow Diagram (Conceptual)

```
Ray
 ↓
Test Sphere 1 → hit at t = 5 ❌ (too far)
 ↓
Test Sphere 2 → hit at t = 2 ✔ (best so far)
 ↓
Test Sphere 3 → hit at t = 3 ❌ (worse than 2)
 ↓
Final answer → Sphere 2
```

---

# 🧩 8. Hit Function Contract (Important Design Idea)

Every object must implement:

$$
\text{hit}(ray, t_{min}, t_{max}) \rightarrow (bool, t, normal, point)
$$

### Meaning:

| Output | Meaning               |
| ------ | --------------------- |
| bool   | did ray hit?          |
| $t$    | distance along ray    |
| point  | intersection position |
| normal | surface direction     |

---

# 🧠 9. Why we use `temp_rec`

We need temporary storage because:

* Each object may overwrite `rec`
* We only want to keep the **best candidate**

So:

```python
temp_rec = HitRecord()
```

acts like a “scratchpad”.

---

# 🚀 10. Summary (Core Insight)

This class is basically implementing:

### 🟢 “Find minimum valid intersection among many objects”

Mathematically:

$$
t^* = \min_{i \in \text{objects}} t_i \quad \text{such that } t_i \in [t_{min}, t_{max}]
$$

---

If you want next step, I can show:

* how `Sphere.hit()` works in Python (full derivation)
* how quadratic equation connects to this
* or how this becomes a full “world renderer”

Just tell 👍


In [3]:
import sys 
sys.path.append("../../Ray Tracing in One Weekend/")

from util.ray import Ray
from util.vec3 import Vec3
from util.color import write_color
from util.vec3 import unit_vector
from util.vec3 import dot
from tqdm import tqdm
from util.hit_record import HitRecord
from util.hittable import Hittable
from util.sphere import Sphere
import math

In [4]:
from typing import List, Optional

# We assume these exist already:
# Ray, HitRecord, Hittable

class HittableList:
    def __init__(self, object=None):
        self.objects: List["Hittable"] = []
        if object is not None:
            self.add(object)

    def clear(self):
        self.objects.clear()

    def add(self, object: "Hittable"):
        self.objects.append(object)

    def hit(self, ray, t_min: float, t_max: float, rec: "HitRecord") -> bool:
        temp_rec = HitRecord()
        hit_anything = False
        closest_so_far = t_max

        for obj in self.objects:
            if obj.hit(ray, t_min, closest_so_far, temp_rec):
                hit_anything = True
                closest_so_far = temp_rec.t
                rec.copy(temp_rec)

        return hit_anything